# SubaRAG: build the serving indexes
Use a **fresh, free Colab runtime**. A free T4 GPU is optional; CPU works but is slower. Do not buy compute units. This notebook reads the already prepared private corpus, not the large upstream Parquet files. It builds all 59,594 pilot passages, not the full MSMARCO-XI dataset. No Gemini or ElevenLabs calls are made.

Upload `SubaRAG-index-source.zip` in the first cell. Keep this runtime open after the build. Only a small report is downloaded; indexes remain in Colab until you explicitly run the optional upload cell. Never upload `.env` files or put tokens in notebook source.

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, io, json, os, subprocess, sys, tempfile, zipfile

def run_logged(arguments):
    with subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    if code != 0:
        raise RuntimeError(f'Build stopped with exit code {code}. Keep this runtime and share the last progress lines, without secrets.')

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one SubaRAG index source ZIP; renamed copies are accepted.')
archive_bytes = next(iter(uploaded.values()))
SOURCE_SHA256 = hashlib.sha256(archive_bytes).hexdigest()
source_root = Path(tempfile.mkdtemp(prefix='subarag-index-source-', dir='/content'))
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    builder = archive.read('SubaRAG/backend/src/subarag/experiments/build_index.py').decode()
    if '--model-revision' not in builder or '--device' not in builder:
        raise ValueError('Old source ZIP: use SubaRAG-index-source.zip from the new index slice.')
    for member in archive.infolist():
        destination = (source_root / member.filename).resolve()
        if not destination.is_relative_to(source_root) or (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('Archive contains an unsafe path or symlink.')
    archive.extractall(source_root)
backend = source_root / 'SubaRAG/backend'
run_logged([sys.executable, '-m', 'pip', 'install', f'{backend}[rag]', 'huggingface_hub>=0.34,<2'])
run_logged([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps', str(backend)])
sys.path.insert(0, str(backend / 'src'))
os.environ['PYTHONPATH'] = str(backend / 'src')
import subarag
if not Path(subarag.__file__).resolve().is_relative_to(backend):
    raise RuntimeError('An older SubaRAG was already imported. Restart the runtime and run this notebook again.')
print('Index source installed; archive SHA-256:', SOURCE_SHA256)


In [ ]:
from getpass import getpass
from subarag.data.remote import fetch_prepared_corpus
from subarag.data.manifest import verify_manifest
from huggingface_hub import HfApi

DATA_REPO = 'silentkirmada/SubaRAG-data'
DATA_REVISION = 'e29f5c990a4fc0d8e05771be4007f1c6560adfc2'
MODEL = 'intfloat/multilingual-e5-small'
WORK = Path(tempfile.mkdtemp(prefix='subarag-index-build-', dir='/content'))
read_token = getpass('HF read token for SubaRAG-data (hidden): ').strip()
try:
    CORPUS_MANIFEST = await fetch_prepared_corpus(repo_id=DATA_REPO, revision=DATA_REVISION, token=read_token, cache_dir=WORK / 'corpus')
finally:
    del read_token
prepared = verify_manifest(CORPUS_MANIFEST)
print('Verified corpus:', prepared.counts.model_dump())
MODEL_REVISION = HfApi(token=False).model_info(MODEL).sha
print('Pinned embedding model:', MODEL, MODEL_REVISION)
BUNDLE = WORK / 'retrieval'
BUNDLE.mkdir()


In [ ]:
import torch
from subarag.retrieval import load_hnsw_artifact
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Building on:', DEVICE, '(no paid services or generation calls)')
for language in ('en', 'hi', 'mr'):
    output = BUNDLE / language
    manifest = output / 'retrieval-manifest.json'
    if manifest.is_file():
        artifact = load_hnsw_artifact(manifest, expected_model=MODEL)
        if artifact.language != language or artifact.model_revision != MODEL_REVISION:
            raise ValueError('Existing index does not match this build; do not mix artifacts.')
        del artifact
        print('Reusing verified completed index:', language)
        continue
    if output.exists() and any(output.iterdir()):
        raise RuntimeError(f'Partial index at {output}. Keep it and share the error; it will not be overwritten.')
    run_logged([sys.executable, '-u', '-m', 'subarag.experiments.build_index',
                '--manifest', str(CORPUS_MANIFEST), '--output', str(output),
                '--language', language, '--model', MODEL, '--model-revision', MODEL_REVISION,
                '--device', DEVICE, '--batch-size', '32'])


In [ ]:
import importlib.metadata

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while chunk := handle.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()

report = {'schema_version': 1, 'artifact_kind': 'subarag_retrieval_bundle',
          'prepared_repo': DATA_REPO, 'prepared_revision': DATA_REVISION,
          'prepared_manifest_sha256': sha256_file(CORPUS_MANIFEST),
          'source_archive_sha256': SOURCE_SHA256, 'model_name': MODEL,
          'model_revision': MODEL_REVISION, 'build_device': DEVICE,
          'packages': {name: importlib.metadata.version(name) for name in ('torch', 'sentence-transformers', 'faiss-cpu', 'numpy')},
          'languages': {}, 'files': {}}
for language in ('en', 'hi', 'mr'):
    manifest = BUNDLE / language / 'retrieval-manifest.json'
    artifact = load_hnsw_artifact(manifest, expected_model=MODEL)
    expected = sum(shard.rows for shard in prepared.shards if shard.kind == 'documents' and shard.language == language)
    if len(artifact.index.documents) != expected or artifact.language != language or artifact.model_revision != MODEL_REVISION:
        raise ValueError(f'Index completeness or provenance check failed for {language}')
    report['languages'][language] = {'documents': expected, 'dimension': artifact.index.index.d, 'corpus_version': artifact.corpus_version}
    del artifact
    for name in ('retrieval-manifest.json', 'documents.jsonl', 'index.faiss'):
        path = BUNDLE / language / name
        report['files'][f'{language}/{name}'] = {'bytes': path.stat().st_size, 'sha256': sha256_file(path)}
report['total_bytes'] = sum(item['bytes'] for item in report['files'].values())
REPORT = BUNDLE / 'bundle.json'
REPORT.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report['languages'], indent=2))
print('Verified bundle size (MiB):', round(report['total_bytes'] / 1024**2, 2))
print('All indexes verified. Keep this runtime open. No upload or deployment has occurred.')
files.download(str(REPORT))


## Stop here and share `bundle.json`
The build is ready for review. Keep the runtime open so the files are not lost. The next cell is an **optional, explicit upload** to the existing private dataset repository, not deployment. It asks for a separate token with write access only to `silentkirmada/SubaRAG-data`; the read-only token used by Render cannot upload. It adds a content-addressed `retrieval/` subdirectory and never replaces the root corpus files. Do not make the repository public. Run it only when ready to upload; even Run all requires typing `UPLOAD`.

In [ ]:
confirmation = input('Type UPLOAD to store the verified indexes in the private HF repository, or press Enter to skip: ')
if confirmation == 'UPLOAD':
    for relative, metadata in report['files'].items():
        path = BUNDLE / relative
        if path.stat().st_size != metadata['bytes'] or sha256_file(path) != metadata['sha256']:
            raise ValueError('An artifact changed after verification; upload cancelled.')
    if json.loads(REPORT.read_text()) != report:
        raise ValueError('The bundle report changed; upload cancelled.')
    bundle_id = sha256_file(REPORT)
    remote_path = f'retrieval/{bundle_id}'
    write_token = getpass('HF upload token scoped to SubaRAG-data (hidden): ').strip()
    api = HfApi(token=write_token)
    try:
        if not api.dataset_info(DATA_REPO).private:
            raise RuntimeError('Artifact repository must remain private.')
        commit = api.upload_folder(repo_id=DATA_REPO, repo_type='dataset', folder_path=str(BUNDLE),
                                   path_in_repo=remote_path, allow_patterns=['bundle.json', *report['files']],
                                   commit_message='Add verified English Hindi Marathi RAG indexes')
        print('Upload complete:', commit.commit_url)
        print('Retrieval revision:', commit.oid)
        print('Retrieval path:', remote_path)
    except Exception:
        raise RuntimeError('Upload failed. Check token write access, private repository visibility, and network; your local files are preserved.') from None
    finally:
        del api, write_token
else:
    print('Skipped upload. Keep this runtime open.')
